# Computation in Nonlinear Codes

This notebook follows the README's six claims for the trained example, **1A, 1B, 1C, 2A, 2B, 2C**. It evaluates all 16 Boolean inputs, using exact rational arithmetic on the saved weights. A failed linear probe is not evidence of unreadability: we verify either a strict affine separator or an intersection of the two convex hulls.

The final sections check the training-attempt counts reported in the README and ask how many of the four bits remain linearly unreadable throughout a successful XOR circuit. These are exhaustive checks of a small finite task, not a claim about generalization.

The example is a jointly refined model: XOR circuit 89 from the XOR-training stage, decoder restart 8, reconstruction weight 1, with the circuit learning at 1/100 of the decoder rate. Its encoder is the stage-1 autoencoder for seed 9. The circuit first learned XOR, then received joint XOR and reconstruction training; it is not a fresh circuit trained jointly from its random initialization.

## Load the saved example

The encoder and both decoders have two ReLU hidden layers and linear outputs, with biases. Each of the four residual blocks computes

$$r_{l+1}=\operatorname{ReLU}(A_lz_l+a_l),\qquad z_{l+1}=z_l+B_lr_{l+1}+b_l.$$

The XOR head is affine, followed by a threshold at zero. The code below uses only the Python standard library; it does not retrain the model or need an LP solver.

In [1]:
import itertools, json, hashlib
from fractions import Fraction as Q
from pathlib import Path

here = Path.cwd()
if not (here / 'results/joint_decoder_refinement/selected-example-exact.json').exists():
    here = here / 'results/computation_in_nonlinear_codes'
example_path = here / 'results/joint_decoder_refinement/selected-example-exact.json'
saved = json.loads(example_path.read_text())

def rational(v):
    return [rational(x) for x in v] if isinstance(v,list) else Q(v)
weights = {m:{k:rational(v) for k,v in sd.items()}
           for m,sd in saved['exact_weights'].items()}
x = list(map(list,itertools.product((0,1),repeat=4)))
y = [[a^b,c^d] for a,b,c,d in x]
assert x == saved['input_patterns']
assert y == saved['xor_targets']

def affine(rows, matrix, bias):
    return [[sum(a*b for a,b in zip(row,w))+c for w,c in zip(matrix,bias)]
            for row in rows]
def plus(a,b):
    return [[u+v for u,v in zip(r,s)] for r,s in zip(a,b)]
def mlp(rows,w):
    for k in (0,2,4):
        rows=affine(rows,w[f'net.{k}.weight'],w[f'net.{k}.bias'])
        if k<4: rows=[[max(Q(0),v) for v in r] for r in rows]
    return rows
def predict(rows):
    return [[int(v>0) for v in row] for row in rows]
def correct(rows, targets):
    return sum(a==b for a,b in zip(predict(rows),targets))
def run(model_weights, modes=None):
    weights=model_weights
    modes=modes or {}
    z=mlp(x,weights['encoder'])
    states,pre,hs=[z],[],[]
    w=weights['transition']
    for k in range(4):
        a=affine(z,w[f'ins.{k}.weight'],w[f'ins.{k}.bias'])
        h=[[Q(0) if modes.get((k+1,j+1))=='zero' else v
            if modes.get((k+1,j+1))=='linear' else max(Q(0),v)
            for j,v in enumerate(row)] for row in a]
        z=plus(z,affine(h,w[f'outs.{k}.weight'],w[f'outs.{k}.bias']))
        states.append(z);pre.append(a);hs.append(h)
    logits=affine(z,weights['head']['weight'],weights['head']['bias'])
    return states,pre,hs,logits,mlp(z,weights['d4'])
states,pre,hs,logits,reconstruction=run(weights)
assert logits==rational(saved['xor_logits'])
assert reconstruction==rational(saved['input_logits'])
print('Example:', example_path.relative_to(here))
print('Training record:', saved['training'])
print('Recomputed all 16 inputs, five residual states, and four ReLU branches.')

def show_table(headers, rows):
    rows = [[str(v) for v in row] for row in rows]
    widths = [max(len(str(h)), *(len(row[j]) for row in rows)) for j,h in enumerate(headers)]
    print(' | '.join(str(h).ljust(w) for h,w in zip(headers,widths)))
    print('-+-'.join('-'*w for w in widths))
    for row in rows: print(' | '.join(v.ljust(w) for v,w in zip(row,widths)))

Example: results/joint_decoder_refinement/selected-example-exact.json
Training record: {'id': 648, 'matched_run': 232, 'encoder_seed': 9, 'lambda': 1.0, 'learning_rate': 0.003000000026077032, 'branch_scale': 0.3, 'decoder_initialization': 'random', 'checkpoint_step': 20000, 'selection_loss': 0.00022581075609195977, 'source_circuit': 89, 'decoder_restart': 8, 'circuit_rate_multiplier': 0.01, 'xor_correct_patterns': 16, 'reconstruction_correct_patterns': 16, 'distinct_final_codes': 16, 'persistent_unreadable_bits': [1, 2], 'persistent_unreadable_count': 2, 'every_xor_has_persistent_nonlinear_input': True, 'six_claims_pass': True, 'all_requirements_pass': True}
Recomputed all 16 inputs, five residual states, and four ReLU branches.


## 1A. Compression: two coordinates for four input features

Check the matrix sizes, including the two coordinates carried by every residual state. The computation receives the encoder's code; it has no additional path from the original four input bits.

In [2]:
def check_mlp_shape(module, widths):
    for k,(n_in,n_out) in zip((0,2,4),zip(widths,widths[1:])):
        matrix=module[f'net.{k}.weight']; bias=module[f'net.{k}.bias']
        assert len(matrix)==len(bias)==n_out
        assert all(len(row)==n_in for row in matrix)
check_mlp_shape(weights['encoder'],[4,4,4,2])
for name in ('d0','d4'): check_mlp_shape(weights[name],[2,4,4,4])
for k in range(4):
    for side in ('ins','outs'):
        assert len(weights['transition'][f'{side}.{k}.bias'])==2
        assert len(weights['transition'][f'{side}.{k}.weight'])==2
        assert all(len(row)==2 for row in weights['transition'][f'{side}.{k}.weight'])
assert len(weights['head']['bias'])==2
assert len(weights['head']['weight'])==2
assert all(len(row)==2 for row in weights['head']['weight'])
assert all(len(row)==2 for state in states for row in state)
print('Encoder 4 -> 4 -> 4 -> 2; decoders 2 -> 4 -> 4 -> 4.')
print('Four residual blocks, each with a two-coordinate state and two ReLU units.')
print('Compression: 2 bottleneck dimensions < 4 input features.')


Encoder 4 -> 4 -> 4 -> 2; decoders 2 -> 4 -> 4 -> 4.
Four residual blocks, each with a two-coordinate state and two ReLU units.
Compression: 2 bottleneck dimensions < 4 input features.


## 1B. Nonlinear encoding: no affine map reproduces these codes

If $z(x)=Wx+b$, then $z(0000)$ fixes $b$. Subtracting it from $z(1000)$ and $z(0100)$ fixes the first two columns of $W$. That forces

$$z(1100)=z(1000)+z(0100)-z(0000).$$

One violation is enough to rule out every affine encoder on the Boolean domain. This is a statement about the actual input-to-code function, not merely about having ReLU layers in the architecture.

In [3]:
z0=states[0]
ids={tuple(row):i for i,row in enumerate(x)}
a,b,c,d=[ids[row] for row in [(0,0,0,0),(1,0,0,0),(0,1,0,0),(1,1,0,0)]]
forced=[u+v-w for u,v,w in zip(z0[b],z0[c],z0[a])]
difference=[actual-predicted for actual,predicted in zip(z0[d],forced)]
assert any(value!=0 for value in difference)
def decimals(row): return '('+', '.join(f'{float(v):.6f}' for v in row)+')'
show_table(['Input or prediction','Code'],[
    ('0000',decimals(z0[a])),('1000',decimals(z0[b])),('0100',decimals(z0[c])),
    ('1100: forced affine prediction',decimals(forced)),('1100: actual encoder',decimals(z0[d])),
    ('Actual minus prediction',decimals(difference))])
print('The mismatch is nonzero in exact arithmetic: no affine encoder matches these codes.')


Input or prediction            | Code                  
-------------------------------+-----------------------
0000                           | (-1.323779, 0.364316) 
1000                           | (-0.563587, -1.823088)
0100                           | (-0.877922, 0.754839) 
1100: forced affine prediction | (-0.117731, -1.432566)
1100: actual encoder           | (-0.123959, -1.604435)
Actual minus prediction        | (-0.006229, -0.171869)
The mismatch is nonzero in exact arithmetic: no affine encoder matches these codes.


The README also quantifies how far the codes are from *any* affine map: fit $z \approx Wx + b$ to all 16 codes by least squares and compare the average distance between each code and its fitted value with the distance between the two closest codes.

In [4]:
def solve(matrix, vector):
    n=len(matrix); a=[row[:]+[v] for row,v in zip(matrix,vector)]
    for col in range(n):
        pivot=next(r for r in range(col,n) if a[r][col]!=0)
        a[col],a[pivot]=a[pivot],a[col]
        for r in range(n):
            if r!=col and a[r][col]!=0:
                f=a[r][col]/a[col][col]
                a[r]=[u-f*v for u,v in zip(a[r],a[col])]
    return [a[i][n]/a[i][i] for i in range(n)]
design=[[Q(v) for v in row]+[Q(1)] for row in x]
gram=[[sum(r[i]*r[j] for r in design) for j in range(5)] for i in range(5)]
fitted=[[Q(0)]*len(z0[0]) for _ in x]
for k in range(len(z0[0])):
    coef=solve(gram,[sum(r[i]*z[k] for r,z in zip(design,z0)) for i in range(5)])
    for i,r in enumerate(design): fitted[i][k]=sum(c*v for c,v in zip(coef,r))
dist=lambda u,v: sum(float(a-b)**2 for a,b in zip(u,v))**0.5
misses=[dist(u,v) for u,v in zip(z0,fitted)]
closest=min(dist(z0[i],z0[j]) for i in range(16) for j in range(i+1,16))
mean_miss=sum(misses)/len(misses)
print(f'Mean distance from each code to the best affine fit: {mean_miss:.3f}')
print(f'Distance between the two closest codes:            {closest:.4f}')
print(f'Ratio: {mean_miss/closest:.1f}')
assert abs(mean_miss-0.41)<0.005 and 9.5<mean_miss/closest<10.5

Mean distance from each code to the best affine fit: 0.408
Distance between the two closest codes:            0.0408
Ratio: 10.0


## 1C. Source preservation: the initial decoder recovers all four bits

Check every bit on every input, with a strictly positive signed logit margin. This also proves that the initial encoder maps the 16 inputs to 16 distinct codes.

In [5]:
def check_readout(name, values, targets):
    assert predict(values)==targets
    margin=min((2*t-1)*v for row,target in zip(values,targets) for v,t in zip(row,target))
    assert margin>0
    print(f'{name}: all outputs correct on 16/16 inputs; minimum signed logit = {float(margin):.6f}')
initial_reconstruction=mlp(states[0],weights['d0'])
check_readout('Initial decoder D0', initial_reconstruction, x)
assert len(set(map(tuple,states[0])))==16

Initial decoder D0: all outputs correct on 16/16 inputs; minimum signed logit = 4.753707


## 2A. The trained head computes both XORs

Both outputs must be correct on the same 16 inputs. We additionally check the final decoder: the computation retains enough information for this particular nonlinear decoder to reconstruct all four source bits.

In [6]:
check_readout('XOR head', logits, y)
check_readout('Final decoder D4', reconstruction, x)
counts=[len(set(map(tuple,state))) for state in states]
assert counts==[16]*5
show_table(['State','Distinct codes'], [(f'z{k}',count) for k,count in enumerate(counts)])

XOR head: all outputs correct on 16/16 inputs; minimum signed logit = 6.252490
Final decoder D4: all outputs correct on 16/16 inputs; minimum signed logit = 6.460817
State | Distinct codes
------+---------------
z0    | 16            
z1    | 16            
z2    | 16            
z3    | 16            
z4    | 16            


## 2B. Neither XOR is linearly readable in the initial code

Here “linearly readable” means perfectly classified by an affine score followed by a threshold. Intersecting convex hulls rule out every such readout: each class has a convex combination that gives exactly the same point. An affine score has the same value on both combinations and therefore cannot strictly separate their labels.

The verifier checks the proposed weights, their signs, their sums, and equality in every coordinate. When a feature is readable, it instead checks a proposed separator's margin on every input.

In [7]:
def verify_certificate(rows, labels, certificate):
    assert certificate['separable'] is not None
    rows=[[Q(v) for v in r] for r in rows]
    if certificate['separable']:
        c=list(map(Q,certificate['coefficients']))
        margins=[(2*t-1)*sum(a*b for a,b in zip(c,r+[Q(1)]))
                 for r,t in zip(rows,labels)]
        assert min(margins)>0
        assert min(margins)==Q(certificate['minimum_exact_margin'])
    else:
        totals=[Q(0),Q(0)]
        points=[[Q(0)]*len(rows[0]) for _ in range(2)]
        for item in certificate['exact_hull_support']:
            i,t,q=item['input_index'],item['label'],Q(item['weight'])
            assert t==labels[i] and q>0
            totals[t]+=q
            points[t]=[a+q*b for a,b in zip(points[t],rows[i])]
        assert totals==[1,1] and points[0]==points[1]


In [8]:
initial_xor=saved['residual_xor_audits'][0]
assert states[0]==rational(initial_xor['exact_activations'])
for j,certificate in enumerate(initial_xor['certificates']):
    verify_certificate(states[0],[row[j] for row in y],certificate)
    assert certificate['separable'] is False
    print(f'XOR {j+1}: verified exact convex-hull intersection in z0.')

XOR 1: verified exact convex-hull intersection in z0.
XOR 2: verified exact convex-hull intersection in z0.


## 2C. Each XOR has an input that stays linearly unreadable

At block $l+1$, check the complete four-dimensional interface $(z_l,r_{l+1})$, because the update has access to both the state and its ReLU branch. Checking only the two-dimensional residual states would miss possible readouts that combine them with the branch.

We check **all four source bits at all four interfaces**, then intersect their unreadable sets. For the main example, this set must contain an operand from each XOR pair. The same hull witnesses also certify the adjacent residual states: those are affine functions of the joint interface.

In [9]:
interface_rows=[]
persistent=set(range(4))
for k,audit in enumerate(saved['interfaces']):
    live=[a+b for a,b in zip(states[k],hs[k])]
    assert len(audit['certificates'])==4 and all(len(row)==4 for row in live)
    assert live==rational(audit['exact_activations'])
    for j,certificate in enumerate(audit['certificates']):
        verify_certificate(live,[row[j] for row in x],certificate)
    persistent &= {j for j,c in enumerate(audit['certificates']) if not c['separable']}
    interface_rows.append((f'(z{k}, r{k+1})', *['Yes' if c['separable'] else 'No' for c in audit['certificates']]))
assert sorted(persistent)==saved['persistent_nonlinear_input_bits']
assert all(persistent & set(pair) for pair in [(0,1),(2,3)])
for k,audit in enumerate(saved['interfaces']):
    for j in persistent:
        for state in (states[k],states[k+1]):
            verify_certificate(state,[row[j] for row in x],audit['certificates'][j])
show_table(['Joint interface','x0 readable','x1 readable','x2 readable','x3 readable'],interface_rows)
print('Persistently unreadable bits:', ', '.join(f'x{j}' for j in sorted(persistent)))
print('At least one operand of each XOR stays unreadable at every joint interface and residual state.')

Joint interface | x0 readable | x1 readable | x2 readable | x3 readable
----------------+-------------+-------------+-------------+------------
(z0, r1)        | Yes         | No          | No          | Yes        
(z1, r2)        | Yes         | No          | No          | No         
(z2, r3)        | Yes         | No          | No          | No         
(z3, r4)        | Yes         | No          | No          | No         
Persistently unreadable bits: x1, x2
At least one operand of each XOR stays unreadable at every joint interface and residual state.


For a concrete witness, inspect one bit at the first joint interface. The displayed values are rounded; the equalities checked in the code are exact.

In [10]:
bit=min(persistent)
certificate=saved['interfaces'][0]['certificates'][bit]
live=[a+b for a,b in zip(states[0],hs[0])]
points=[[Q(0)]*4 for _ in range(2)]
totals=[Q(0),Q(0)]; rows=[]
for item in certificate['exact_hull_support']:
    i,t,weight=item['input_index'],item['label'],Q(item['weight'])
    totals[t]+=weight
    points[t]=[a+weight*b for a,b in zip(points[t],live[i])]
    rows.append((f'{i:04b}',t,f'{float(weight):.6f}'))
assert totals==[1,1] and points[0]==points[1]
show_table(['Input',f'x{bit}','Convex weight'],rows)
print(f'Weighted average for x{bit}=0:',decimals(points[0]))
print(f'Weighted average for x{bit}=1:',decimals(points[1]))
print('The two four-dimensional points are exactly equal.')


Input | x1 | Convex weight
------+----+--------------
0101  | 1  | 0.090383     
1100  | 1  | 0.909617     
0000  | 0  | 0.047317     
1000  | 0  | 0.641020     
1010  | 0  | 0.311663     
Weighted average for x1=0: (-0.166924, -1.383301, 0.000000, 0.085811)
Weighted average for x1=1: (-0.166924, -1.383301, 0.000000, 0.085811)
The two four-dimensional points are exactly equal.


All six claims now have direct checks. This does not establish that this trained architecture is minimal, that unreadable bits have no useful linear correlations, or that other networks must compute the same way.

## Earlier training stages

The README's table starts with two earlier stages. **Initial autoencoder:** ten width-4 autoencoders (seeds 0–9), each the best of 3 restarts, were trained on reconstruction alone; we count those that reconstruct all 16 inputs. **XOR training:** with the two successful encoders frozen, 64 circuits per encoder were trained on the XORs alone. The persistently-unreadable screen for these circuits is numerical (floating point); exact certificates for the 13 circuits that continue to the refinement study come from that study's frozen-circuit control below.

In [11]:
from collections import Counter
stage1={}
for seed in range(10):
    path=here/'results'/'width4_codec'/f'width-4-seed-{seed}-compressor.json'
    if path.exists():
        stage1[seed]=json.loads(path.read_text())['input_exact']
    else:  # seed 1 was saved only as part of its full run
        stage1[seed]=json.loads((here/'results'/'width4_codec'/f'width-4-seed-{seed}.json').read_text())['initial_input_exact']
show_table(['Seed','Fraction of inputs reconstructed'],[(s,f'{v:.4f}') for s,v in stage1.items()])
perfect=[s for s,v in stage1.items() if v==1.0]
assert perfect==[4,9]
print(f'Initial autoencoder: {len(perfect)}/10 seeds reconstruct all 16 inputs (seeds {perfect}).')

task=json.loads((here/'results'/'focused_stage2'/'task-only.json').read_text())
screen=json.loads((here/'results'/'focused_stage2'/'task-only-input-screen.json').read_text())
total=task['runs_per_code']*len(task['encoder_seeds'])
assert total==128 and len(task['perfect_task_indices'])==51
assert sorted(r['model_index'] for r in screen)==sorted(task['perfect_task_indices'])
unreadable=Counter(len(r['persistent_nonlinear_inputs_numerical']) for r in screen)
each_pair=sorted(r['model_index'] for r in screen if r['every_pair_numerical'])
assert unreadable=={2:48,1:3} and len(each_pair)==13
refinement_protocol=json.loads((here/'results'/'joint_decoder_refinement'/'protocol.json').read_text())
assert each_pair==refinement_protocol['source_circuits']
print(f'XOR training: {len(screen)}/{total} circuits solve both XORs; '
      f'{unreadable[2]} have two persistently unreadable bits and {unreadable[1]} have one (numerical screen).')
print(f'{len(each_pair)} keep an operand of each XOR unreadable; these are the refinement study\'s source circuits.')

Seed | Fraction of inputs reconstructed
-----+---------------------------------
0    | 0.6250                          
1    | 0.8125                          
2    | 0.4375                          
3    | 0.6250                          
4    | 1.0000                          
5    | 0.5000                          
6    | 0.9375                          
7    | 0.4375                          
8    | 0.6875                          
9    | 1.0000                          
Initial autoencoder: 2/10 seeds reconstruct all 16 inputs (seeds [4, 9]).
XOR training: 51/128 circuits solve both XORs; 48 have two persistently unreadable bits and 3 have one (numerical screen).
13 keep an operand of each XOR unreadable; these are the refinement study's source circuits.


## Training attempts in the refinement study

Rerun commands are in [`REPRODUCING.md`](../../REPRODUCING.md).

The refinement study starts from the 13 XOR circuits above, all on encoder 9, and gives each one 32 decoder initializations for every reconstruction weight $\beta \in \{0, 1, 3, 10\}$ (stored as `lambda` in the saved files). The joint conditions train on all 16 inputs with

$$L=L_{\mathrm{XOR}}+\beta L_{\mathrm{reconstruction}},$$

where each term is mean binary cross-entropy over its outputs and inputs, updating the circuit at 1/100 of the decoder learning rate. At $\beta=0$ the circuit stays frozen and only the decoder is fit. The README's example uses $\beta=1$. The same decoder initializations are reused across weights, and the fixed decoder normalization is folded into the saved first layer.

Each run contributes exactly one checkpoint, checked every 250 steps including the endpoints. The joint conditions maximize the number of inputs with all six outputs correct, then the sum of XOR-correct and reconstruction-correct pattern counts, then minimize loss; the frozen control maximizes reconstruction accuracy, then minimizes reconstruction loss. No readability term enters these choices. Counts below are recomputed from the saved weights, and the denominator always includes failed runs. These runs refine circuits that already solve the XORs; they are not independent circuit trainings.

A bit is **persistently unreadable** if it is not perfectly linearly separable at any of the four joint interfaces. The histogram is conditional on both XOR outputs being correct on every input; it includes all four bits, not only the two operands highlighted in the README.

In [12]:
from collections import Counter
sweep_names=['joint_decoder_refinement']
sweeps=[]
audit_counts=Counter()
for name in sweep_names:
    directory=here/'results'/name
    summary=json.loads((directory/'summary.json').read_text())
    source=directory/'trained-weights.json'
    assert hashlib.sha256(source.read_bytes()).hexdigest()==summary['weights_source_sha256']
    data=json.loads(source.read_text())
    assert data['protocol']==summary['protocol']==json.loads((directory/'protocol.json').read_text())
    encoders={seed:{module:{key:rational(value) for key,value in sd.items()} for module,sd in modules.items()}
              for seed,modules in data['encoders'].items()}
    initial=json.loads((directory/'encoder-audits.json').read_text())
    for seed,modules in encoders.items():
        encoded=mlp(x,modules['encoder'])
        assert predict(mlp(encoded,modules['d0']))==x
        assert encoded[12]!=[a+b-c for a,b,c in zip(encoded[8],encoded[4],encoded[0])]
        for j,cert in enumerate(initial[seed]['initial_input_certificates']):
            verify_certificate(encoded,[row[j] for row in x],cert)
        assert [c['separable'] for c in initial[seed]['initial_input_certificates']]==({'4':[True,True,False,False],'9':[True,False,False,True]}[seed])
        for j,cert in enumerate(initial[seed]['initial_xor_certificates']):
            verify_certificate(encoded,[row[j] for row in y],cert)
            assert cert['separable'] is False
    assert len(data['runs'])==len(summary['runs'])==summary['protocol']['total_training_runs']
    for run_row,record in zip(data['runs'],summary['runs']):
        assert all(record[key]==value for key,value in run_row.items() if key not in ('transition','head','d4'))
        modules={**encoders[str(run_row['encoder_seed'])],
                 **{module:{key:rational(value) for key,value in run_row[module].items()}
                    for module in ('transition','head','d4')}}
        zs,_,branches,scores,reconstructed=run(modules)
        xor_count=correct(scores,y); reconstruction_count=correct(reconstructed,x)
        assert xor_count==record['xor_correct_patterns']
        assert reconstruction_count==record['reconstruction_correct_patterns']
        assert len(set(map(tuple,zs[-1])))==record['distinct_final_codes']
        hidden=set()
        audit_path=directory/'audits'/f"run-{run_row['id']:03d}.json"
        if xor_count==16 and not audit_path.exists():
            # Certificates for this run are not included in this repository; use the audited summary record.
            audit_counts['from summary']+=1
            hidden=set(record['persistent_unreadable_bits'])
        elif xor_count==16:
            audit_counts['verified here']+=1
            certificates=json.loads(audit_path.read_text())
            assert certificates['weights_source_sha256']==summary['weights_source_sha256']
            hidden=set(range(4))
            assert len(certificates['interface_certificates'])==4
            for k,block in enumerate(certificates['interface_certificates']):
                joint=[a+b for a,b in zip(zs[k],branches[k])]
                assert len(block)==4
                for j,cert in enumerate(block):
                    verify_certificate(joint,[row[j] for row in x],cert)
                hidden &= {j for j,c in enumerate(block) if c['separable'] is False}
            assert sorted(hidden)==record['persistent_unreadable_bits']
            assert len(hidden)==record['persistent_unreadable_count']
        else:
            assert record['persistent_unreadable_count'] is None
        each_pair=xor_count==16 and all(hidden & set(pair) for pair in [(0,1),(2,3)])
        assert each_pair==record['six_claims_pass']==record['every_xor_has_persistent_nonlinear_input']
        assert (each_pair and reconstruction_count==16)==record['all_requirements_pass']
    sweeps.append((name,summary))
    print(f"Verified {name}: {len(data['runs'])} checkpoints recomputed from the saved weights.")

Verified joint_decoder_refinement: 1664 checkpoints recomputed from the saved weights.


In [13]:
count_rows=[]; histogram_rows=[]; claim_rows=[]
for name,summary in sweeps:
    for arm in summary['arms']:
        records=[r for r in summary['runs'] if r['lambda']==arm['lambda']]
        good=[r for r in records if r['xor_correct_patterns']==16]
        both=[r for r in good if r['reconstruction_correct_patterns']==16]
        histogram=Counter(r['persistent_unreadable_count'] for r in good)
        both_histogram=Counter(r['persistent_unreadable_count'] for r in both)
        expected={'runs':len(records),'xor_correct':len(good),'both_correct':len(both),
                  'reconstruction_correct':sum(r['reconstruction_correct_patterns']==16 for r in records),
                  'distinct_final_codes':sum(r['distinct_final_codes']==16 for r in records),
                  'six_claims_pass':sum(r['six_claims_pass'] for r in records),
                  'all_requirements_pass':sum(r['all_requirements_pass'] for r in records)}
        assert all(arm[key]==value for key,value in expected.items())
        assert arm['unreadable_bit_histogram_among_xor_correct']=={str(k):histogram[k] for k in range(5)}
        assert arm['unreadable_bit_histogram_among_both_correct']=={str(k):both_histogram[k] for k in range(5)}
        assert arm['unresolved_xor_successes']==0
        label=f"{arm['lambda']:g}"
        count_rows.append((label,len(records),len(good),expected['reconstruction_correct'],len(both),expected['all_requirements_pass']))
        histogram_rows.append((label,len(good),*[histogram[k] for k in range(5)]))
        claim_rows.append((label,len(records),len(records),len(records),len(good),len(records),expected['six_claims_pass']))
show_table(['β','Runs','XORs 16/16','D4 16/16','Both 16/16','Both + 2C'],count_rows)
print('\nUnreadable-bit counts among perfect XOR circuits:')
show_table(['β','XOR successes','0 bits','1 bit','2 bits','3 bits','4 bits'],histogram_rows)
print('\nIndividual claim counts (2C shown among XOR successes):')
show_table(['β','1A','1B','1C','2A','2B','2C'],claim_rows)
print(f"\nCertificates: {audit_counts['verified here']} XOR-successful runs verified from their audit files; "
      f"{audit_counts['from summary']} taken from the audited summary because their audit files are not included in this repository.")

β  | Runs | XORs 16/16 | D4 16/16 | Both 16/16 | Both + 2C
---+------+------------+----------+------------+----------
0  | 416  | 416        | 1        | 1          | 1        
1  | 416  | 403        | 3        | 3          | 3        
3  | 416  | 380        | 1        | 1          | 1        
10 | 416  | 255        | 3        | 3          | 3        

Unreadable-bit counts among perfect XOR circuits:
β  | XOR successes | 0 bits | 1 bit | 2 bits | 3 bits | 4 bits
---+---------------+--------+-------+--------+--------+-------
0  | 416           | 0      | 0     | 416    | 0      | 0     
1  | 403           | 0      | 3     | 400    | 0      | 0     
3  | 380           | 0      | 12    | 368    | 0      | 0     
10 | 255           | 0      | 16    | 239    | 0      | 0     

Individual claim counts (2C shown among XOR successes):
β  | 1A  | 1B  | 1C  | 2A  | 2B  | 2C 
---+-----+-----+-----+-----+-----+----
0  | 416 | 416 | 416 | 416 | 416 | 416
1  | 416 | 416 | 416 | 403 | 416 | 400
3  |

### Which starting circuits were successfully reconstructed?

Count distinct starting circuits separately from decoder restarts. Repeated successes for one circuit or across loss weights do not become new independent circuit discoveries.

In [14]:
refinement_rows=[]
for name,summary in sweeps:
    for arm in summary['arms']:
        good=[r for r in summary['runs'] if r['lambda']==arm['lambda'] and r['all_requirements_pass']]
        source_ids=sorted({r['source_circuit'] for r in good})
        for source,group in arm['by_source_circuit'].items():
            records=[r for r in summary['runs'] if r['lambda']==arm['lambda'] and r['source_circuit']==int(source)]
            assert len(records)==group['runs']==32
            assert sum(r['all_requirements_pass'] for r in records)==group['all_requirements_pass']
        refinement_rows.append((f"{arm['lambda']:g}",len(good),len(source_ids),source_ids))
show_table(['β','Successful restarts','Distinct circuits','Source circuits'],refinement_rows)
assert saved['training']['source_circuit']==89 and saved['training']['decoder_restart']==8
assert saved['training']['lambda']==1 and saved['training']['circuit_rate_multiplier']==0.01

β  | Successful restarts | Distinct circuits | Source circuits
---+---------------------+-------------------+----------------
0  | 1                   | 1                 | [71]           
1  | 3                   | 2                 | [87, 89]       
3  | 1                   | 1                 | [89]           
10 | 3                   | 3                 | [66, 71, 89]   


Encoder 9 already passes 1A, 1B, 1C and 2B, so those columns are fixed by the setup. It initially exposes two bits, so three or four persistently unreadable bits are impossible here. The "Both + 2C" column additionally requires a perfect final decoder and an unreadable operand in each XOR pair.

If all 16 final codes are distinct, some inverse on this finite set exists. Failure to train this particular width-4 decoder is not evidence of an information-theoretic collision. Likewise, these counts do not isolate representational capacity from optimization difficulty.

Reproduction: `joint_decoder_refinement.py` runs the refinement study and `audit_joint_reconstruction.py` evaluates each checkpoint and constructs exact certificates. Use a fresh output directory when rerunning.